# 10 — Pilot-based GPU time and resource planner

**Purpose:** Estimate quota needs from REAL pilot timings; report assumptions instead of pretending estimates were observed full-run durations.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# GitHub setup: run this as the FIRST code cell in Kaggle or Colab.
# This clones the source code only; GPU checkpoints/results need separate storage.
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
REPO_FOLDER = "nlp_research_proj"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # True: update an existing clean clone from GitHub

if Path("/kaggle/working").is_dir():
    workspace = Path("/kaggle/working")
elif Path("/content").is_dir():
    workspace = Path("/content")
else:
    workspace = Path.cwd()

repo_dir = workspace / REPO_FOLDER
project_dir = repo_dir / PROJECT_FOLDER

# Local development support: use the existing source folder outside Kaggle/Colab.
local_project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_utils.py").is_file()), None)
if local_project is not None and not (Path("/kaggle/working").is_dir() or Path("/content").is_dir()):
    ROOT = local_project
else:
    if not repo_dir.exists():
        print("Cloning:", REPO_URL)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    elif PULL_LATEST:
        # Only fast-forward; protects existing local changes from being overwritten.
        subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    else:
        print("Using existing clone:", repo_dir)

    ROOT = project_dir

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(
        f"Cannot find project_utils.py at {ROOT}. Confirm the GitHub folder name."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print("Project loaded from:", ROOT)


Full fine-tuning cannot be assigned one universal duration. Batch size, GPU model, token length, optimizer, and startup overhead matter. **No timing value below is a published benchmark until your own pilot runs.**

In [ ]:
import math, pandas as pd
pilots=[r for r in read_jsonl(ROOT/'outputs/metrics/train_runs.jsonl') if r.get('pilot')]
if not pilots:
    print('No pilot measurements yet. Run GPU pilot in notebook 00 first.')
else:
    out=[]
    for r in pilots:
        for examples_per_class in [10,25,50]:
            n=7*examples_per_class
            effective_batch=cfg()['per_device_batch_size']*cfg()['gradient_accumulation']
            estimated_steps=math.ceil(n/effective_batch)*cfg()['train_epochs']
            # Estimate from completed actual training timings. Warmup bias is possible.
            sps=r['wall_seconds']/max(r['global_steps'],1)
            out.append(dict(pilot=r['run_id'], method=r['method'],per_class=examples_per_class,
                            approx_optimizer_steps=estimated_steps,naive_minutes=round(sps*estimated_steps/60,1),
                            warmup_included=True,estimated_from_pilot_steps=r['global_steps']))
    display(pd.DataFrame(out))
    print('Use 10–20 steps for a steadier speed measurement; 2-step pilots include heavy warmup.')

### Practical quota and checkpoint advice

- For 2–3-epoch runs, start at 10/class and move to 25/class only after both methods work.
- Save checkpoints after **each finished run** to persistent storage (Google Drive on Colab, Kaggle dataset/output).
- A free notebook session can disconnect; saved files in ephemeral workspace may disappear.
- If you need to preserve checkpoints through interruptions, implement step-based checkpoints and resume in a dedicated long-run notebook. This project currently defaults to saving only completed runs to reduce storage.
- A 2-step pilot alone does not measure model download time or full training duration reliably.
- Measure `nvidia-smi` and peak `torch.cuda.max_memory_allocated()`; the latter is allocated memory, not full driver-level device utilization.